# Illumina WGS age × sex counts (for manuscript Figure 1A)

Run in a **Verily Workbench** Jupyter app with CDR BigQuery access and the
`vwb-aou-datasets-controlled-v9` workspace mount.

**What it does**
1. Reads Illumina WGS `person_id`s from the v9 CRAM manifest (not the CDR alone).
2. Joins those ids to CDR demographics: survey `sex_at_birth` and age at the CDR cutoff.
3. Writes an aggregated count table (no person ids): `age`, `sex`, `n`.

**Outputs** (under `~/illumina_age_sex/` by default)
- `illumina_wgs_age_sex_counts.tsv` — what the manuscript figure loader needs
- `illumina_wgs_age_sex_summary.txt` — totals and small-cell notes

Age is capped at 90 (90+), matching panel A. Cells with `n < 20` are zeroed in the
exported TSV (AoU small-cell rule) but reported in the summary.

Demographics come from the **CDR BigQuery dataset**, not from
`vwb-aou-datasets-controlled-v9` (that bucket only has genomics files).

In [ ]:
from __future__ import annotations

import os
import textwrap
from pathlib import Path

import pandas as pd
from google.cloud import bigquery

# ---- edit these if needed ----
OUT_DIR = Path.home() / "illumina_age_sex"
MANIFEST_CANDIDATES = [
    Path.home() / "workspace" / "vwb-aou-datasets-controlled-v9" / "v9" / "wgs" / "cram" / "manifest.csv",
    Path("workspace/vwb-aou-datasets-controlled-v9/v9/wgs/cram/manifest.csv"),
]
# Survey concept for "Sex at birth" in All of Us CDRs.
SEX_AT_BIRTH_SOURCE_CONCEPT_ID = 1585849
MIN_CELL = 20
AGE_CAP = 90
# Leave blank to auto-detect from WORKSPACE_CDR / GOOGLE_PROJECT.
CDR_OVERRIDE = ""
# Optional hard override for the CDR reference date used for age_at_cdr.
# Leave blank to try cdr_metadata / dataset labels, then fall back to CURRENT_DATE().
CDR_CUTOFF_DATE = ""  # e.g. "2024-10-01"

OUT_DIR.mkdir(parents=True, exist_ok=True)
COUNTS_TSV = OUT_DIR / "illumina_wgs_age_sex_counts.tsv"
SUMMARY_TXT = OUT_DIR / "illumina_wgs_age_sex_summary.txt"
IDS_CSV = OUT_DIR / "illumina_wgs_person_ids.csv"

print("OUT_DIR:", OUT_DIR)

## 1. Resolve CDR dataset and CRAM manifest

In [ ]:
def resolve_cdr() -> str:
    if CDR_OVERRIDE.strip():
        return CDR_OVERRIDE.strip()
    for key in ("WORKSPACE_CDR", "CDR_DATASET_ID", "BIGQUERY_DATASET"):
        value = (os.environ.get(key) or "").strip()
        if value:
            print(f"CDR from ${key}: {value}")
            return value
    raise RuntimeError(
        "No CDR dataset in the environment. Set WORKSPACE_CDR or CDR_OVERRIDE "
        "to project.dataset (e.g. fc-aou-cdr-prod-ct.C2024Q3R1)."
    )


def resolve_manifest() -> Path:
    for path in MANIFEST_CANDIDATES:
        if path.is_file():
            return path.resolve()
    raise FileNotFoundError(
        "Could not find v9 WGS CRAM manifest. Expected one of:\n  "
        + "\n  ".join(str(p) for p in MANIFEST_CANDIDATES)
    )


CDR = resolve_cdr()
MANIFEST = resolve_manifest()
PROJECT = (
    os.environ.get("GOOGLE_CLOUD_PROJECT")
    or os.environ.get("GOOGLE_PROJECT")
    or os.environ.get("GCP_PROJECT")
    or None
)
client = bigquery.Client(project=PROJECT)
print("CDR:", CDR)
print("PROJECT:", PROJECT or client.project)
print("MANIFEST:", MANIFEST)

## 2. Load Illumina WGS person ids from the CRAM manifest

These are the short-read WGS samples in the controlled v9 release.

In [ ]:
manifest = pd.read_csv(MANIFEST, dtype=str)
id_col = next(
    (c for c in ("person_id", "research_id", "sample_id") if c in manifest.columns),
    None,
)
if id_col is None:
    raise RuntimeError(f"manifest has no person id column; columns={list(manifest.columns)}")

wgs_ids = (
    manifest[id_col]
    .astype(str)
    .str.strip()
    .loc[lambda s: s.ne("") & s.str.lower().ne("nan")]
    .drop_duplicates()
)
wgs_ids.to_frame("person_id").to_csv(IDS_CSV, index=False)
print(f"Illumina WGS person ids: {len(wgs_ids):,}  (from {id_col})")
print("wrote", IDS_CSV)

## 3. Probe CDR tables and resolve the age cutoff date

In [ ]:
def table_exists(table: str) -> bool:
    sql = f"""
    SELECT 1
    FROM `{CDR}`.INFORMATION_SCHEMA.TABLES
    WHERE table_name = @table
    LIMIT 1
    """
    job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("table", "STRING", table)]
        ),
    )
    return next(job.result(), None) is not None


def columns_of(table: str) -> set[str]:
    sql = f"""
    SELECT column_name
    FROM `{CDR}`.INFORMATION_SCHEMA.COLUMNS
    WHERE table_name = @table
    """
    job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("table", "STRING", table)]
        ),
    )
    return {row.column_name for row in job.result()}


HAS_PERSON = table_exists("person")
HAS_OBS = table_exists("observation")
HAS_CDR_META = table_exists("cdr_metadata") or table_exists("_cdr_metadata")
PERSON_COLS = columns_of("person") if HAS_PERSON else set()
print("person:", HAS_PERSON, "columns sample:", sorted(PERSON_COLS)[:20])
print("observation:", HAS_OBS)
print("cdr_metadata:", HAS_CDR_META)

if not HAS_PERSON:
    raise RuntimeError(f"`{CDR}.person` not found; check WORKSPACE_CDR")
if "birth_datetime" not in PERSON_COLS and "year_of_birth" not in PERSON_COLS:
    raise RuntimeError("person table has neither birth_datetime nor year_of_birth")

In [ ]:
def resolve_cutoff_date() -> str:
    if CDR_CUTOFF_DATE.strip():
        print("CDR cutoff from CDR_CUTOFF_DATE:", CDR_CUTOFF_DATE.strip())
        return CDR_CUTOFF_DATE.strip()

    for table in ("cdr_metadata", "_cdr_metadata"):
        if not table_exists(table):
            continue
        cols = columns_of(table)
        for col in ("cdr_cutoff_date", "cutoff_date", "date", "value"):
            if col not in cols:
                continue
            sql = f"SELECT CAST({col} AS STRING) AS d FROM `{CDR}.{table}` LIMIT 5"
            try:
                rows = list(client.query(sql).result())
            except Exception as exc:  # noqa: BLE001
                print(f"skip {table}.{col}: {exc}")
                continue
            for row in rows:
                text = (row.d or "").strip()
                # Prefer ISO dates.
                if len(text) >= 10 and text[4] == "-" and text[7] == "-":
                    print(f"CDR cutoff from {table}.{col}: {text[:10]}")
                    return text[:10]
    print("CDR cutoff not found in metadata; using CURRENT_DATE() in BigQuery")
    return ""


CUTOFF = resolve_cutoff_date()
if CUTOFF:
    AGE_DATE_SQL = f"DATE('{CUTOFF}')"
else:
    AGE_DATE_SQL = "CURRENT_DATE()"

## 4. Upload WGS ids and aggregate age × sex in BigQuery

Sex is survey **sex at birth** (`observation_source_concept_id = 1585849`), not DRAGEN ploidy.
Age matches the long-read `age_at_cdr` convention: floor of day difference / 365.25, capped at 90.

In [ ]:
ids_table = f"{client.project}.scratch_illumina_age_sex.wgs_person_ids"
client.delete_table(ids_table, not_found_ok=True)

# Create dataset if needed.
dataset_ref = bigquery.Dataset(f"{client.project}.scratch_illumina_age_sex")
dataset_ref.location = os.environ.get("BIGQUERY_LOCATION") or None
try:
    client.create_dataset(dataset_ref, exists_ok=True)
except Exception as exc:  # noqa: BLE001
    print("create_dataset note:", exc)

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema=[bigquery.SchemaField("person_id", "STRING")],
)
with IDS_CSV.open("rb") as handle:
    load_job = client.load_table_from_file(handle, ids_table, job_config=job_config)
load_job.result()
print("uploaded ids ->", ids_table, "rows", client.get_table(ids_table).num_rows)

In [ ]:
if "birth_datetime" in PERSON_COLS:
    age_expr = f"""
      LEAST(
        CAST(FLOOR(DATE_DIFF({AGE_DATE_SQL}, DATE(p.birth_datetime), DAY) / 365.25) AS INT64),
        {AGE_CAP}
      )
    """
else:
    # Coarser fallback when only year_of_birth is present.
    age_expr = f"""
      LEAST(
        EXTRACT(YEAR FROM {AGE_DATE_SQL}) - CAST(p.year_of_birth AS INT64),
        {AGE_CAP}
      )
    """

if HAS_OBS:
    sex_cte = f"""
    sex AS (
      SELECT
        CAST(person_id AS STRING) AS person_id,
        -- Prefer the survey answer string when present.
        COALESCE(
          NULLIF(value_source_value, ''),
          NULLIF(value_as_string, ''),
          CAST(value_as_concept_id AS STRING)
        ) AS sex_raw
      FROM `{CDR}.observation`
      WHERE observation_source_concept_id = {SEX_AT_BIRTH_SOURCE_CONCEPT_ID}
      QUALIFY ROW_NUMBER() OVER (
        PARTITION BY person_id
        ORDER BY observation_date DESC, observation_datetime DESC
      ) = 1
    )
    """
    sex_join = "LEFT JOIN sex s USING (person_id)"
    sex_select = "s.sex_raw"
elif "sex_at_birth_source_value" in PERSON_COLS:
    sex_cte = f"""
    sex AS (
      SELECT CAST(person_id AS STRING) AS person_id,
             sex_at_birth_source_value AS sex_raw
      FROM `{CDR}.person`
    )
    """
    sex_join = "LEFT JOIN sex s USING (person_id)"
    sex_select = "s.sex_raw"
else:
    raise RuntimeError(
        "Could not find sex_at_birth in observation (1585849) or person.sex_at_birth_source_value"
    )

sql = f"""
WITH
wgs AS (
  SELECT DISTINCT CAST(person_id AS STRING) AS person_id
  FROM `{ids_table}`
),
{sex_cte},
demo AS (
  SELECT
    w.person_id,
    {age_expr} AS age,
    {sex_select} AS sex_raw
  FROM wgs w
  INNER JOIN `{CDR}.person` p
    ON CAST(p.person_id AS STRING) = w.person_id
  {sex_join}
),
clean AS (
  SELECT
    age,
    CASE
      WHEN LOWER(sex_raw) IN ('female', 'f') THEN 'Female'
      WHEN LOWER(sex_raw) IN ('male', 'm') THEN 'Male'
      WHEN sex_raw IN ('GenderIdentity_Man', 'MAN') THEN 'Male'
      WHEN sex_raw IN ('GenderIdentity_Woman', 'WOMAN') THEN 'Female'
      -- AoU PMI / PPI coded answers commonly seen in extracts:
      WHEN sex_raw LIKE 'SexAtBirth_Female%' OR sex_raw = 'Female' THEN 'Female'
      WHEN sex_raw LIKE 'SexAtBirth_Male%' OR sex_raw = 'Male' THEN 'Male'
      WHEN sex_raw LIKE '%Female%' THEN 'Female'
      WHEN sex_raw LIKE '%Male%' AND sex_raw NOT LIKE '%Female%' THEN 'Male'
      ELSE sex_raw
    END AS sex
  FROM demo
  WHERE age IS NOT NULL AND age >= 0
)
SELECT age, sex, COUNT(*) AS n
FROM clean
GROUP BY 1, 2
ORDER BY 1, 2
"""

print(textwrap.shorten(sql, width=400, placeholder=" ..."))
counts = client.query(sql).result().to_dataframe(create_bqstorage_client=False)
print(counts.head(20))
print("rows", len(counts), "total n", int(counts["n"].sum()))

## 5. Inspect raw sex labels, suppress small cells, write outputs

In [ ]:
print("Sex label totals before small-cell suppression:")
print(counts.groupby("sex", dropna=False)["n"].sum().sort_values(ascending=False))

usable = counts[counts["sex"].isin(["Female", "Male"])].copy()
usable["age"] = usable["age"].astype(int)
usable["n"] = usable["n"].astype(int)

small = usable[usable["n"] < MIN_CELL].copy()
export = usable.copy()
export.loc[export["n"] < MIN_CELL, "n"] = 0

export = (
    export[export["n"] > 0]
    .sort_values(["age", "sex"])
    .reset_index(drop=True)
)
export.to_csv(COUNTS_TSV, sep="\t", index=False)

other = counts[~counts["sex"].isin(["Female", "Male"])].copy()
summary = []
summary.append(f"CDR={CDR}")
summary.append(f"manifest={MANIFEST}")
summary.append(f"cutoff={CUTOFF or 'CURRENT_DATE()'}")
summary.append(f"wgs_ids={len(wgs_ids)}")
summary.append(f"counted_female_male={int(usable['n'].sum())}")
summary.append(f"exported_after_min_cell_{MIN_CELL}={int(export['n'].sum())}")
summary.append(f"small_cells={len(small)} suppressed_n={int(small['n'].sum()) if len(small) else 0}")
if len(other):
    summary.append("non_female_male_sex_totals:")
    for sex, n in other.groupby("sex")["n"].sum().sort_values(ascending=False).items():
        summary.append(f"  {sex!r}: {int(n)}")
SUMMARY_TXT.write_text("\n".join(summary) + "\n")

print("\n".join(summary))
print("\nwrote", COUNTS_TSV)
print("wrote", SUMMARY_TXT)
display(export.head(12))

## 6. Download for the manuscript

Copy `illumina_wgs_age_sex_counts.tsv` out of Workbench (UI download, `gsutil cp`, etc.) into:

`aou-lr-phase-2/illumina_wgs_age_sex_counts.tsv`

or tell the figure script where it lives. Also paste the total `exported_after_min_cell_*`
number into the panel A caption (replacing the provisional ~550,000).